# Module 2: Mathematics for Deep Learning
### The Mathematical Foundation of Neural Networks

**Learning Level:** B.Tech 1st Year → PhD Researchers  
**Estimated Time:** 4-5 hours  
**Prerequisites:** Basic calculus, linear algebra awareness

## Learning Outcomes

1. Understand vectors and matrices — core data structures for neural networks
2. Perform matrix operations — multiplication, transposes, decompositions
3. Compute derivatives — single and partial, using chain rule
4. Calculate gradients — foundation of gradient descent optimization
5. Work with probability — distributions and cross-entropy loss
6. Interpret softmax — convert scores to probabilities
7. Visualize concepts — geometry behind linear algebra and calculus

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
from scipy.stats import norm, multivariate_normal
from scipy.special import softmax as scipy_softmax
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("All libraries imported successfully!")


---
## 1. Vectors and Matrices

### 1.1 Intuition

A **vector** is a list of numbers. Example: a person described as [height, weight, age] = [180, 75, 25]

A **matrix** is a 2D grid of numbers. Think of it as stacked vectors.

### 1.2 Why This Matters

Neural networks process data as vectors and matrices. Every operation is linear algebra.

In [ ]:

# Vector example
v = np.array([1, 2, 3])
print(f"Vector: {v}")
print(f"Shape: {v.shape}")
print(f"Magnitude (L2 norm): {np.linalg.norm(v):.2f}")

# Matrix example
M = np.array([[1, 2], [3, 4], [5, 6]])
print(f"\nMatrix shape: {M.shape}")
print(f"Matrix:\n{M}")
print(f"\nTranspose:\n{M.T}")


---
## 2. Matrix Multiplication

### 2.1 Why It Matters

Every neural network layer is matrix multiplication: output = W @ input + bias

### 2.2 The Rule

If A is (m × n) and B is (n × p), then C = A @ B is (m × p).

In [ ]:

# Matrix multiplication
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])
C = A @ B

print(f"A shape: {A.shape}")
print(f"B shape: {B.shape}")
print(f"C shape: {C.shape}")
print(f"\nC = A @ B:")
print(C)
print(f"\nVerify: C[0,0] = (1×5) + (2×7) = {1*5 + 2*7}")


---
## 3. Dot Product

### 3.1 Definition

For vectors u and v: u · v = sum(u[i] * v[i])

### 3.2 Geometric Meaning

u · v = ||u|| * ||v|| * cos(θ)

If θ = 0° (same direction): maximum  
If θ = 90° (perpendicular): zero  
If θ = 180° (opposite): minimum

In [ ]:

u = np.array([1, 2, 3])
v = np.array([4, 5, 6])
dot_product = np.dot(u, v)

print(f"u = {u}")
print(f"v = {v}")
print(f"u · v = {dot_product}")
print(f"Manual: (1*4) + (2*5) + (3*6) = {1*4 + 2*5 + 3*6}")

# Orthogonal vectors
u_orth = np.array([1, 0])
v_orth = np.array([0, 1])
print(f"\nOrthogonal vectors u={u_orth}, v={v_orth}")
print(f"u · v = {np.dot(u_orth, v_orth)} (perpendicular)")


---
## 4. Linear Transformations

A linear transformation T satisfies:
- T(u + v) = T(u) + T(v) (additivity)
- T(αu) = αT(u) (scaling)

Every linear transformation = matrix multiplication

In [ ]:

# Rotation transformation (45 degrees)
theta = np.radians(45)
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta), np.cos(theta)]])

v = np.array([1, 0])
v_rotated = R @ v

print(f"Original vector: {v}")
print(f"After 45° rotation: {v_rotated}")
print(f"Vector is rotated but not scaled ✓")

# Scaling transformation
S = np.array([[2, 0], [0, 3]])
v_scaled = S @ v
print(f"\nAfter scaling by (2,3): {v_scaled}")
print(f"Vector is scaled (stretched) ✓")


---
## 5. Eigenvalues (Intuition)

For some special vectors v and values λ: A @ v = λ * v

v is an **eigenvector** (unchanged direction)  
λ is an **eigenvalue** (scaling factor)

### Why It Matters

Eigenvalues > 1 cause gradients to EXPLODE  
Eigenvalues < 1 cause gradients to VANISH

In [ ]:

# Eigenvalue decomposition
A = np.array([[2, 1], [1, 2]])
eigenvalues, eigenvectors = np.linalg.eig(A)

print(f"Matrix A:\n{A}")
print(f"\nEigenvalues: {eigenvalues}")
print(f"\nEigenvectors:\n{eigenvectors}")

# Verify: A @ v = λ * v
for i in range(len(eigenvalues)):
    v = eigenvectors[:, i]
    Av = A @ v
    lambda_v = eigenvalues[i] * v
    print(f"\nEigenvector {i+1}: λ = {eigenvalues[i]:.2f}")
    print(f"  A @ v = {Av}")
    print(f"  λ @ v = {lambda_v}")
    print(f"  Match: {np.allclose(Av, lambda_v)}")


---
## 6. Derivatives

The derivative f'(x) measures rate of change of f at point x.

Definition: f'(x) = lim(h→0) [f(x+h) - f(x)] / h

### In Deep Learning

Derivative tells how much the loss changes if we change a weight slightly.

In [ ]:

def f(x):
    return x**2

def f_derivative_analytical(x):
    return 2*x

def f_derivative_numerical(x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2*h)

x_test = 3.0
analytical = f_derivative_analytical(x_test)
numerical = f_derivative_numerical(x_test)

print(f"Function: f(x) = x²")
print(f"At x = {x_test}:")
print(f"  f(x) = {f(x_test)}")
print(f"  f'(x) analytical = {analytical}")
print(f"  f'(x) numerical = {numerical:.6f}")
print(f"  Error: {abs(analytical - numerical):.2e}")


---
## 7. Partial Derivatives

When a function has multiple inputs, ∂f/∂x is derivative w.r.t x only, treating others as constants.

### In Deep Learning

∂Loss/∂W tells how loss changes if we tweak weight W

In [ ]:

# Function: f(x, y) = x^2 + xy + y^2
def f_multi(x, y):
    return x**2 + x*y + y**2

def partial_dx(x, y, h=1e-5):
    return (f_multi(x+h, y) - f_multi(x-h, y)) / (2*h)

def partial_dy(x, y, h=1e-5):
    return (f_multi(x, y+h) - f_multi(x, y-h)) / (2*h)

x, y = 1.0, 2.0
print(f"Function: f(x,y) = x² + xy + y²")
print(f"At point ({x}, {y}):")
print(f"  f(x,y) = {f_multi(x, y)}")
print(f"  ∂f/∂x = {partial_dx(x, y):.4f}")
print(f"  ∂f/∂y = {partial_dy(x, y):.4f}")


---
## 8. Chain Rule

If y = f(g(x)), then: dy/dx = df/dg · dg/dx

Multiply the derivatives!

### In Deep Learning

Backpropagation = applying chain rule through the network

In [ ]:

# Function composition: y = (3x + 2)^2
def g(x):
    return 3*x + 2

def f(u):
    return u**2

def composite(x):
    return f(g(x))

# Chain rule: df/dx = df/du * du/dx
def chain_rule_derivative(x):
    u = g(x)
    df_du = 2*u      # derivative of u^2
    du_dx = 3        # derivative of 3x+2
    return df_du * du_dx

x_test = 2.0
print(f"Composite function: y = (3x + 2)²")
print(f"At x = {x_test}:")
print(f"  y = {composite(x_test)}")
print(f"  dy/dx (chain rule) = {chain_rule_derivative(x_test)}")
print(f"\nVerification (expand: y = 9x² + 12x + 4):")
print(f"  dy/dx = 18x + 12 = 18({x_test}) + 12 = {18*x_test + 12}")


---
## 9. Gradients

The gradient ∇f = [∂f/∂x, ∂f/∂y, ...] is a vector of all partial derivatives.

It points in the direction of steepest INCREASE.

### In Deep Learning

Gradient descent: move in NEGATIVE gradient direction to minimize loss

In [ ]:

# Gradient descent on f(x,y) = (x-1)^2 + (y-1)^2
def f_2d(x, y):
    return (x - 1)**2 + (y - 1)**2

def gradient_2d(x, y):
    grad_x = 2*(x - 1)
    grad_y = 2*(y - 1)
    return np.array([grad_x, grad_y])

# Start from (3, 3)
x, y = 3.0, 3.0
learning_rate = 0.1

print(f"Function: f(x,y) = (x-1)² + (y-1)²")
print(f"Minimum at: (1, 1) with value 0")
print(f"\nGradient Descent:")

for iteration in range(20):
    loss = f_2d(x, y)
    grad = gradient_2d(x, y)
    x -= learning_rate * grad[0]
    y -= learning_rate * grad[1]
    
    if iteration % 5 == 0:
        print(f"  Iter {iteration:2d}: ({x:.3f}, {y:.3f}), Loss: {loss:.4f}")

print(f"\nConverged to: ({x:.4f}, {y:.4f})")
print(f"Actual minimum: (1.0000, 1.0000)")


---
## 10. Probability Basics

Probability P(event) is between 0 and 1.

P(A) = 0: never happens  
P(A) = 0.5: equally likely  
P(A) = 1: always happens

In [ ]:

# Probability example
from scipy.stats import norm

# Standard normal distribution
x = np.linspace(-3, 3, 1000)
y = norm.pdf(x)

print("Standard Normal Distribution:")
print(f"P(x <= 0) = {norm.cdf(0):.4f}")
print(f"P(-1 <= x <= 1) = {norm.cdf(1) - norm.cdf(-1):.4f}")
print(f"P(-2 <= x <= 2) = {norm.cdf(2) - norm.cdf(-2):.4f}")


---
## 11. Gaussian Distribution

The bell curve: P(x) = (1 / (σ√(2π))) * exp(-(x - μ)² / (2σ²))

μ: mean (center)  
σ: standard deviation (width)

In [ ]:

from scipy.stats import norm

# Different Gaussians
x = np.linspace(-5, 5, 1000)

print("Probability of different ranges in standard normal (μ=0, σ=1):")
print(f"  P(x <= 0) = {norm.cdf(0):.4f}")
print(f"  P(x <= 1) = {norm.cdf(1):.4f}")
print(f"  P(x <= 2) = {norm.cdf(2):.4f}")
print(f"  P(x <= 3) = {norm.cdf(3):.4f}")

# Shifted Gaussian
print(f"\nGaussian with μ=5, σ=1:")
print(f"  P(x <= 5) = {norm.cdf(5, loc=5, scale=1):.4f}")


---
## 12. Logarithms

Logarithm is the inverse of exponentiation.

If 2³ = 8, then log₂(8) = 3  
If e^x = y, then ln(y) = x

### In Deep Learning

Cross-entropy loss uses logarithms

In [ ]:

# Logarithm examples
print("Logarithm Examples:")
print(f"log₂(8) = {np.log2(8):.4f}")
print(f"log₁₀(100) = {np.log10(100):.4f}")
print(f"ln(e) = {np.log(np.e):.4f}")
print(f"ln(1) = {np.log(1):.4f}")

# Properties
a, b = 10, 100
print(f"\nLogarithm Properties:")
print(f"log(10 × 100) = {np.log10(a*b):.4f}")
print(f"log(10) + log(100) = {np.log10(a) + np.log10(b):.4f}")
print(f"Match: {np.isclose(np.log10(a*b), np.log10(a) + np.log10(b))}")


---
## 13. Softmax

Softmax converts raw scores to probabilities.

Softmax(z_i) = e^(z_i) / sum(e^(z_j))

Output always sums to 1 (valid probability distribution)

In [ ]:

from scipy.special import softmax as scipy_softmax

# Example: 3 classes with different scores
scores = np.array([2.0, 1.0, 0.1])
probs = scipy_softmax(scores)

print("Softmax Example:")
print(f"Raw scores: {scores}")
print(f"Softmax probs: {probs}")
print(f"Sum: {probs.sum():.4f}")

# Manual verification
exp_scores = np.exp(scores)
manual = exp_scores / exp_scores.sum()
print(f"\nManual: {manual}")
print(f"Match: {np.allclose(probs, manual)}")

# Interpretation
print(f"\nInterpretation:")
print(f"  Class 0: {probs[0]:.1%} confidence")
print(f"  Class 1: {probs[1]:.1%} confidence")
print(f"  Class 2: {probs[2]:.1%} confidence")


---
## 14. Interview Preparation

### Q1: What is the difference between a vector and a matrix?

**A:** A vector is 1D (a list). A matrix is 2D (a grid). Technically, a vector is a matrix with 1 row or 1 column.

### Q2: Why do we need gradients in neural networks?

**A:** Gradients tell us how much the loss changes if we change a weight. We use this to update weights in the right direction.

### Q3: What is the chain rule and why is it important?

**A:** Chain rule lets us compute derivatives of composite functions: dy/dx = dy/du · du/dx. It's the foundation of backpropagation.

### Q4: What does softmax do?

**A:** Softmax converts raw scores to probabilities that sum to 1. Essential for multi-class classification.

### Q5: How are eigenvalues related to gradient flow?

**A:** Eigenvalues > 1 cause gradients to explode (multiply > 1 each layer). Eigenvalues < 1 cause vanishing gradients.

---
## 15. Mini Project: Gradient Descent from Scratch

Minimize: f(x, y) = (x-3)² + (y-2)²

Start at (0, 0) using gradient descent.

In [ ]:

import matplotlib.pyplot as plt

# Function
def f(x, y):
    return (x - 3)**2 + (y - 2)**2

def grad_f(x, y):
    return np.array([2*(x - 3), 2*(y - 2)])

# Gradient descent
x, y = 0.0, 0.0
learning_rate = 0.1
history = [(x, y)]

print(f"Starting point: ({x}, {y}), Loss: {f(x, y):.2f}")
print(f"Target: (3, 2), Loss: 0.00")
print(f"\nIteration history:")

for i in range(50):
    grad = grad_f(x, y)
    x -= learning_rate * grad[0]
    y -= learning_rate * grad[1]
    history.append((x, y))
    
    if (i+1) % 10 == 0:
        print(f"  Iter {i+1:2d}: ({x:.4f}, {y:.4f}), Loss: {f(x,y):.6f}")

print(f"\nFinal: ({x:.4f}, {y:.4f})")
print(f"Converged: {abs(x-3) < 0.01 and abs(y-2) < 0.01}")

# Visualize
history = np.array(history)
fig, ax = plt.subplots(figsize=(10, 8))

# Contour plot
x_range = np.linspace(-1, 4, 100)
y_range = np.linspace(-1, 4, 100)
X, Y = np.meshgrid(x_range, y_range)
Z = (X-3)**2 + (Y-2)**2

contour = ax.contourf(X, Y, Z, levels=20, cmap='viridis', alpha=0.8)
ax.contour(X, Y, Z, levels=10, colors='white', alpha=0.3)

# Optimization path
ax.plot(history[:, 0], history[:, 1], 'ro-', markersize=3, linewidth=1, alpha=0.7, label='Optimization path')
ax.plot(history[0, 0], history[0, 1], 'bs', markersize=10, label='Start')
ax.plot(3, 2, 'g*', markersize=20, label='Minimum')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Gradient Descent Optimization', fontweight='bold')
ax.legend()
plt.colorbar(contour, ax=ax, label='Loss')
plt.tight_layout()
plt.show()


---
## 16. Challenge Exercises

### Easy
1. Compute: [1, 2, 3] · [4, 5, 6]
2. Create a 2×3 matrix and compute its transpose
3. Verify that (A^T)^T = A

### Medium
1. Compute ∂/∂x of f(x) = 3x² + 2x + 1 at x=2
2. Implement gradient descent for f(x) = x²
3. Compute eigenvalues of [[2, 1], [1, 2]]

### Hard
1. Implement backpropagation for a simple neural network
2. Derive the chain rule for nested functions
3. Show that softmax is invariant to constant shifts: softmax(z + c) = softmax(z)

---
## 17. Summary

You now understand the mathematical foundations of deep learning:

- **Linear Algebra:** Matrices, transformations, eigenvalues
- **Calculus:** Derivatives, chain rule, gradients
- **Probability:** Distributions, softmax, cross-entropy
- **Optimization:** Gradient descent

These concepts underpin everything in neural networks. Master them, and you'll understand the field deeply.

### Next Module
Module 3: Neural Network Fundamentals (Perceptrons, Activation Functions, Loss Functions)

---
## 18. References

### Books
1. Mathematics for Machine Learning - mml-book.github.io
2. Deep Learning (Goodfellow) - Chapter 2-4
3. Linear Algebra Done Right (Axler)

### Courses
1. 3Blue1Brown - Essence of Linear Algebra (YouTube)
2. MIT 18.06 - Linear Algebra (OCW)
3. Stanford CS229 - Machine Learning

### Key Papers
1. Rumelhart et al. (1986) - Backpropagation
2. Glorot & Bengio (2010) - Weight Initialization

---

*Module 2 Complete!*